# Lesson 07 — Similarity & Bipartite Graphs

## Learning Goal

Discover that entities of two different types relate to each other in patterns. Learn to find similar entities, detect duplicates, and recommend connections.

By the end of this lesson, you will:
- Understand bipartite (two-mode) network structure
- Compute node similarity based on shared neighborhoods
- Project bipartite graphs to single-mode and interpret results
- Detect duplicate records using similarity + attribute matching
- Identify single points of failure and consolidation opportunities
- Generate recommendations based on similarity

**Duration**: 90 minutes

**Why this matters**: B2B companies often have duplicates, power users, and niche customers. Bipartite analysis reveals cross-sell opportunities and customer consolidation. Similarity metrics let you find matches without manual review.

**Key Insight**: "In bipartite networks, relationships are mediated through the other mode. Customers aren't directly connected—they're connected through products they buy. Same products = similar customers."

## Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

# Add src directory to path
project_root = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.spatial.distance import jaccard, cosine
from scipy.sparse import csr_matrix

# Load config or construct path manually
try:
    from src.config import Config
    DATA_DIR = Config.PROJECT_ROOT / 'data' / 'seed' / 'sales_accounts'
except ImportError:
    notebook_dir = Path.cwd()
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir.parent.parent if (notebook_dir.parent.parent / 'data').exists() else notebook_dir
    DATA_DIR = project_root / 'data' / 'seed' / 'sales_accounts'

def display_csv_head(df, name, n=5):
    """Display DataFrame info and head."""
    print(f"\n{name}:")
    print(f"  Shape: {df.shape} (rows, columns)")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  First {n} rows:")
    print(df.head(n).to_string(index=False))

# Load datasets
customers_df = pd.read_csv(DATA_DIR / 'customers.csv')
products_df = pd.read_csv(DATA_DIR / 'products.csv')
purchases_df = pd.read_csv(DATA_DIR / 'purchases.csv')

print("Data loaded successfully!")
print(f"  Customers: {len(customers_df)} records")
print(f"  Products: {len(products_df)} records")
print(f"  Purchases: {len(purchases_df)} edges (customer → product)")

## 1. Understanding Bipartite Networks

A **bipartite graph** has two types of nodes. Edges only connect nodes of different types.

**In this dataset**:
- **Customers** (one type): 250 B2B customers
- **Products** (other type): 120 SaaS products
- **Purchases** (edges only between types): Customer buys product (1200 edges)
- **NO customer-customer edges**: Customers don't "connect" directly
- **NO product-product edges**: Products don't "connect" directly

**Key Insight**: "Customers seem unrelated until you look through the product lens."

In [ ]:
# Examine each dataset
display_csv_head(customers_df, "Customers (sample)", n=8)
display_csv_head(products_df, "Products (sample)", n=8)
display_csv_head(purchases_df, "Purchases (sample)", n=10)

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(f"\nCustomers by status:")
print(customers_df['status'].value_counts())

print(f"\nCustomers by industry:")
print(customers_df['industry'].value_counts())

print(f"\nCustomers by company size:")
print(customers_df['company_size'].value_counts())

print(f"\nAccount value distribution:")
print(customers_df['account_value'].describe())

print(f"\nProducts by category:")
print(products_df['category'].value_counts())

print(f"\nProducts by price tier:")
print(products_df['price_tier'].value_counts())

print(f"\nPurchase renewal status:")
print(purchases_df['renewal_status'].value_counts())

## 2. Building the Bipartite Graph

In [ ]:
# Create bipartite graph
G = nx.Graph()  # Bipartite networks are typically undirected

# Add customer nodes with type
for idx, row in customers_df.iterrows():
    G.add_node(
        row['customer_id'],
        node_type='customer',
        company_name=row['company_name'],
        industry=row['industry'],
        status=row['status'],
        account_value=row['account_value']
    )

# Add product nodes with type
for idx, row in products_df.iterrows():
    G.add_node(
        row['product_id'],
        node_type='product',
        product_name=row['product_name'],
        category=row['category'],
        price_tier=row['price_tier']
    )

# Add purchase edges
for idx, row in purchases_df.iterrows():
    G.add_edge(
        row['customer_id'],
        row['product_id'],
        purchase_date=row['purchase_date'],
        quantity=row['quantity'],
        total_value=row['total_value'],
        renewal_status=row['renewal_status']
    )

# Verify bipartite
is_bipartite = nx.is_bipartite(G)

print(f"Bipartite Graph Summary:")
print(f"  Nodes: {G.number_of_nodes()} (customers + products)")
print(f"    - Customers: {len(customers_df)}")
print(f"    - Products: {len(products_df)}")
print(f"  Edges: {G.number_of_edges()} (purchases)")
print(f"  Is bipartite: {is_bipartite}")
print(f"  Bipartite density: {G.number_of_edges() / (len(customers_df) * len(products_df)):.4f}")

if is_bipartite:
    print(f"  ✓ Verified: Only customers connect to products (and vice versa)")
else:
    print(f"  ✗ WARNING: Graph is not bipartite (has same-type edges)")

## 3. Node Similarity in Bipartite Graphs

**Question**: How similar are two customers? Answer: Compare the products they buy.

**Metrics**:
- **Jaccard**: |A ∩ B| / |A ∪ B| (0-1, higher = more similar)
- **Cosine**: |A ∩ B| / sqrt(|A| * |B|) (normalized)
- **Common neighbors**: Raw count of shared products
- **Adamic-Adar**: Sum(1/log(degree(product))) for shared products (rarer products = stronger signal)

In [ ]:
def compute_similarity_metrics(purchases_df):
    """
    Compute pairwise similarity matrix for all customers.
    
    Returns:
        similarity_data: List of (cust1, cust2, jaccard, cosine, common) tuples
    """
    customer_ids = purchases_df['customer_id'].unique()
    
    # Create customer→product sets
    cust_products = {}
    for cust_id in customer_ids:
        cust_products[cust_id] = set(purchases_df[purchases_df['customer_id'] == cust_id]['product_id'].tolist())
    
    # Product degrees (for Adamic-Adar)
    product_degrees = purchases_df['product_id'].value_counts().to_dict()
    
    similarity_data = []
    
    for cust1, cust2 in combinations(customer_ids, 2):
        products1 = cust_products[cust1]
        products2 = cust_products[cust2]
        
        if len(products1) == 0 or len(products2) == 0:
            continue
        
        # Intersection and union
        intersection = len(products1 & products2)
        union = len(products1 | products2)
        
        # Jaccard similarity
        jaccard_sim = intersection / union if union > 0 else 0
        
        # Cosine similarity
        cosine_sim = intersection / np.sqrt(len(products1) * len(products2)) if len(products1) * len(products2) > 0 else 0
        
        # Common neighbors
        common = intersection
        
        # Adamic-Adar
        adamic_adar = 0
        if intersection > 0:
            for product in (products1 & products2):
                degree = product_degrees.get(product, 1)
                adamic_adar += 1 / np.log(degree + 1)
        
        similarity_data.append({
            'customer_1': cust1,
            'customer_2': cust2,
            'jaccard': jaccard_sim,
            'cosine': cosine_sim,
            'common_neighbors': common,
            'adamic_adar': adamic_adar
        })
    
    return pd.DataFrame(similarity_data)

print("Computing customer similarity...")
similarity_df = compute_similarity_metrics(purchases_df)
print(f"Computed {len(similarity_df)} customer pairs")

# Show statistics
print(f"\nSimilarity statistics:")
print(f"  Jaccard:        mean={similarity_df['jaccard'].mean():.3f}, max={similarity_df['jaccard'].max():.3f}")
print(f"  Cosine:         mean={similarity_df['cosine'].mean():.3f}, max={similarity_df['cosine'].max():.3f}")
print(f"  Common products: mean={similarity_df['common_neighbors'].mean():.1f}, max={int(similarity_df['common_neighbors'].max())}")
print(f"  Adamic-Adar:    mean={similarity_df['adamic_adar'].mean():.3f}, max={similarity_df['adamic_adar'].max():.3f}")

In [ ]:
# Find most similar customer pairs
print("\nMost Similar Customer Pairs (by Jaccard similarity):")
print(f"{'Customer 1':<15} {'Customer 2':<15} {'Jaccard':<12} {'Common':<10} {'Company 1':<30}")
print("-" * 85)

top_similar = similarity_df.nlargest(10, 'jaccard')

for _, row in top_similar.iterrows():
    cust1_name = customers_df[customers_df['customer_id'] == row['customer_1']]['company_name'].values[0]
    cust2_name = customers_df[customers_df['customer_id'] == row['customer_2']]['company_name'].values[0]
    
    print(f"{row['customer_1']:<15} {row['customer_2']:<15} {row['jaccard']:<12.3f} {int(row['common_neighbors']):<10} {cust1_name:<30}")

print(f"\n💡 Insight: High Jaccard similarity (>0.7) suggests likely duplicate customers!")

## 4. One-Mode Projections

**Customer projection**: Connect customers who buy the same products.
- Edge = two customers share at least one product
- Edge weight = number of shared products

In [ ]:
# Extract customer and product node sets
customer_nodes = set(c for c in G.nodes() if G.nodes[c].get('node_type') == 'customer')
product_nodes = set(p for p in G.nodes() if G.nodes[p].get('node_type') == 'product')

print(f"Customer nodes: {len(customer_nodes)}")
print(f"Product nodes: {len(product_nodes)}")

# Project to customer graph
G_customer = nx.bipartite.projected_graph(G, customer_nodes, multigraph=False)

# Add edge weights (number of shared products)
for cust1, cust2 in G_customer.edges():
    products1 = set(G.neighbors(cust1)) & product_nodes
    products2 = set(G.neighbors(cust2)) & product_nodes
    shared = len(products1 & products2)
    G_customer[cust1][cust2]['weight'] = shared

print(f"\nCustomer Projection:")
print(f"  Nodes: {G_customer.number_of_nodes()}")
print(f"  Edges: {G_customer.number_of_edges()}")
print(f"  Density: {nx.density(G_customer):.4f}")
print(f"\nBipartite vs Projection:")
print(f"  Bipartite edges: {G.number_of_edges()}")
print(f"  Projection edges: {G_customer.number_of_edges()}")
print(f"  Inflation factor: {G_customer.number_of_edges() / G.number_of_edges():.2f}x")
print(f"\n⚠️ Projection creates many edges! Use edge weights to filter.")

In [ ]:
# Analyze edge weights in projection
edge_weights = [G_customer[u][v]['weight'] for u, v in G_customer.edges()]

print("Edge Weight Distribution (Projected Customer Graph):")
print(f"  Mean shared products: {np.mean(edge_weights):.2f}")
print(f"  Median: {np.median(edge_weights):.0f}")
print(f"  Max: {np.max(edge_weights):.0f}")
print(f"\n  Weight distribution:")
for weight in sorted(set(edge_weights))[:10]:
    count = sum(1 for w in edge_weights if w == weight)
    pct = 100 * count / len(edge_weights)
    print(f"    {weight} shared products: {count} edges ({pct:.1f}%)")

# High-weight edges (strong connections)
strong_edges = [(u, v, w) for u, v, w in G_customer.edges(data='weight') if w >= 5]
print(f"\nStrong customer pairs (5+ shared products): {len(strong_edges)}")

if strong_edges:
    print(f"\n  Top strong pairs:")
    for u, v, w in sorted(strong_edges, key=lambda x: x[2], reverse=True)[:5]:
        name1 = customers_df[customers_df['customer_id'] == u]['company_name'].values[0]
        name2 = customers_df[customers_df['customer_id'] == v]['company_name'].values[0]
        print(f"    {u} ({name1}) — {v} ({name2}): {w} shared products")

## 5. Product Projection & Co-Purchase Analysis

**Product projection**: Connect products frequently bought by the same customers.
- Edge = two products bought by at least one customer
- Edge weight = number of customers buying both

In [ ]:
# Project to product graph
G_product = nx.bipartite.projected_graph(G, product_nodes, multigraph=False)

# Add edge weights (number of shared customers)
for prod1, prod2 in G_product.edges():
    custs1 = set(G.neighbors(prod1)) & customer_nodes
    custs2 = set(G.neighbors(prod2)) & customer_nodes
    shared = len(custs1 & custs2)
    G_product[prod1][prod2]['weight'] = shared

print(f"Product Projection:")
print(f"  Nodes: {G_product.number_of_nodes()}")
print(f"  Edges: {G_product.number_of_edges()}")
print(f"  Density: {nx.density(G_product):.4f}")

# Analyze co-purchase patterns
print(f"\nProduct Co-Purchase Statistics:")
co_purchase_weights = [G_product[u][v]['weight'] for u, v in G_product.edges()]
print(f"  Mean customers per pair: {np.mean(co_purchase_weights):.2f}")
print(f"  Median: {np.median(co_purchase_weights):.0f}")
print(f"  Max: {np.max(co_purchase_weights):.0f}")

# Find strong product affinity pairs
strong_product_edges = [(u, v, w) for u, v, w in G_product.edges(data='weight') if w >= 10]
print(f"\nProduct pairs bought together by 10+ customers: {len(strong_product_edges)}")

if strong_product_edges:
    print(f"\n  Top co-purchased products:")
    for u, v, w in sorted(strong_product_edges, key=lambda x: x[2], reverse=True)[:5]:
        name1 = products_df[products_df['product_id'] == u]['product_name'].values[0]
        cat1 = products_df[products_df['product_id'] == u]['category'].values[0]
        name2 = products_df[products_df['product_id'] == v]['product_name'].values[0]
        cat2 = products_df[products_df['product_id'] == v]['category'].values[0]
        print(f"    {name1} ({cat1})")
        print(f"      ↔ {name2} ({cat2})")
        print(f"      Bought together by {w} customers\n")

## 6. Duplicate Detection

**Goal**: Find likely duplicate customer records using:
1. High product similarity (Jaccard > 0.8)
2. Fuzzy company name matching
3. Industry alignment

In [ ]:
def fuzzy_string_match(s1, s2, threshold=3):
    """
    Simple edit distance for name matching.
    Returns: edit distance (lower = more similar)
    """
    s1 = s1.lower().replace(' ', '')
    s2 = s2.lower().replace(' ', '')
    
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    
    if len(s1) == 0:
        return len(s2)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

print("Detecting Duplicate Customer Candidates...\n")

# Step 1: High similarity pairs
high_sim_pairs = similarity_df[similarity_df['jaccard'] > 0.75].copy()
print(f"Step 1: High similarity pairs (Jaccard > 0.75): {len(high_sim_pairs)}")

if len(high_sim_pairs) > 0:
    duplicate_candidates = []
    
    for _, row in high_sim_pairs.iterrows():
        cust1_id = row['customer_1']
        cust2_id = row['customer_2']
        
        cust1_row = customers_df[customers_df['customer_id'] == cust1_id].iloc[0]
        cust2_row = customers_df[customers_df['customer_id'] == cust2_id].iloc[0]
        
        name1 = cust1_row['company_name']
        name2 = cust2_row['company_name']
        
        # Fuzzy name matching
        edit_dist = fuzzy_string_match(name1, name2)
        name_match_score = 1 / (1 + edit_dist)  # 0-1 score
        
        # Industry match
        industry_match = cust1_row['industry'] == cust2_row['industry']
        
        # Combined score
        combined_score = (
            row['jaccard'] * 0.5 +  # Product overlap is strongest signal
            name_match_score * 0.3 +
            (1 if industry_match else 0) * 0.2
        )
        
        duplicate_candidates.append({
            'customer_1': cust1_id,
            'customer_2': cust2_id,
            'company_1': name1,
            'company_2': name2,
            'industry_1': cust1_row['industry'],
            'industry_2': cust2_row['industry'],
            'value_1': cust1_row['account_value'],
            'value_2': cust2_row['account_value'],
            'jaccard': row['jaccard'],
            'edit_distance': edit_dist,
            'name_match': name_match_score,
            'industry_match': industry_match,
            'combined_score': combined_score
        })
    
    dup_df = pd.DataFrame(duplicate_candidates).sort_values('combined_score', ascending=False)
    
    print(f"\nDuplicate Candidates (sorted by confidence):")
    print(f"{'Cust1':<12} {'Cust2':<12} {'Company 1':<25} {'Company 2':<25} {'Score':<8} {'Match':<8}")
    print("-" * 100)
    
    for _, row in dup_df.head(10).iterrows():
        match_type = "✓ Name" if row['edit_distance'] <= 3 else "Similarity"
        print(f"{row['customer_1']:<12} {row['customer_2']:<12} {row['company_1']:<25} {row['company_2']:<25} {row['combined_score']:<8.3f} {match_type:<8}")
    
    print(f"\n💡 High-confidence duplicates (score > 0.7): {len(dup_df[dup_df['combined_score'] > 0.7])}")
else:
    print("No high-similarity pairs found.")

## 7. Customer Archetypes

Classify customers into types based on behavior patterns.

In [ ]:
# Build customer metrics
metrics_data = []

for cust_id in customers_df['customer_id']:
    cust_row = customers_df[customers_df['customer_id'] == cust_id].iloc[0]
    
    # Purchase info
    products_bought = purchases_df[purchases_df['customer_id'] == cust_id]['product_id'].unique()
    n_products = len(products_bought)
    
    # Product diversity (Shannon entropy across categories)
    categories = products_df[products_df['product_id'].isin(products_bought)]['category'].tolist()
    if len(categories) > 0:
        category_dist = pd.Series(categories).value_counts(normalize=True)
        diversity = -np.sum(category_dist * np.log(category_dist + 1e-10))
    else:
        diversity = 0
    
    # Purchase span (days between first and last purchase)
    if len(products_bought) > 0:
        cust_purchases = purchases_df[purchases_df['customer_id'] == cust_id]['purchase_date']
        first_date = pd.to_datetime(cust_purchases.min())
        last_date = pd.to_datetime(cust_purchases.max())
        span_days = (last_date - first_date).days
    else:
        span_days = 0
    
    metrics_data.append({
        'customer_id': cust_id,
        'company_name': cust_row['company_name'],
        'status': cust_row['status'],
        'account_value': cust_row['account_value'],
        'products_count': n_products,
        'product_diversity': diversity,
        'purchase_span_days': span_days
    })

metrics_df = pd.DataFrame(metrics_data)

print("Customer Metrics Summary:")
print(metrics_df[['products_count', 'product_diversity', 'purchase_span_days', 'account_value']].describe())

In [ ]:
# Classify archetypes
print("\nCustomer Archetypes:\n")

# Archetype 1: Power Users
power_users = metrics_df[metrics_df['products_count'] >= 50]
print(f"🔶 POWER USERS ({len(power_users)}):")
print(f"  Buying 50+ products, high account value")
if len(power_users) > 0:
    print(f"  Top power user: {power_users.iloc[0]['company_name']} ({power_users.iloc[0]['products_count']} products, ${power_users.iloc[0]['account_value']:,.0f})")
print()

# Archetype 2: Loyal Specialists
loyal = metrics_df[
    (metrics_df['products_count'] >= 10) & 
    (metrics_df['products_count'] < 50) &
    (metrics_df['product_diversity'] > 1.0)
]
print(f"📚 LOYAL SPECIALISTS ({len(loyal)}):")
print(f"  10-50 products, diverse portfolio")
if len(loyal) > 0:
    print(f"  Example: {loyal.iloc[0]['company_name']} ({loyal.iloc[0]['products_count']} products)")
print()

# Archetype 3: Niche Customers
niche = metrics_df[(metrics_df['products_count'] < 5)]
print(f"🎯 NICHE CUSTOMERS ({len(niche)}):")
print(f"  1-4 products, focused use case")
if len(niche) > 0:
    print(f"  Example: {niche.iloc[0]['company_name']} ({niche.iloc[0]['products_count']} product(s))")
print()

# Archetype 4: At-Risk
at_risk = metrics_df[metrics_df['status'] == 'at_risk']
print(f"⚠️ AT-RISK ({len(at_risk)}):")
print(f"  Status flagged as at_risk")
if len(at_risk) > 0:
    print(f"  Example: {at_risk.iloc[0]['company_name']} (${at_risk.iloc[0]['account_value']:,.0f})")
print()

# Archetype 5: Inactive
inactive = metrics_df[metrics_df['status'] == 'inactive']
print(f"❌ INACTIVE ({len(inactive)}):")
print(f"  No longer active")
print(f"  Potential for reactivation or churn analysis")

## 8. Critical Nodes Analysis

**Single points of failure**: Niche customers and niche products with low redundancy.

In [ ]:
# Products with few customers (single point of failure)
product_customers = purchases_df.groupby('product_id').size().reset_index(name='customer_count')
rare_products = product_customers[product_customers['customer_count'] <= 2]

print(f"Single-Expert Products (2 or fewer customers):")
print(f"  Count: {len(rare_products)}")
print(f"\n  {' | '.join(['Product', 'Customers', 'Category', 'Price Tier'])}")
print("-" * 70)

for _, row in rare_products.head(10).iterrows():
    prod_row = products_df[products_df['product_id'] == row['product_id']].iloc[0]
    print(f"{row['product_id']:12} {row['customer_count']:>9} {prod_row['category']:20} {prod_row['price_tier']}")

if len(rare_products) > 0:
    print(f"\n⚠️ These products are vulnerable—if the customer leaves, product loses its base!")
print()

# Customers with few products (vulnerable)
singleton_customers = metrics_df[metrics_df['products_count'] == 1]
print(f"Single-Product Customers (using only 1 product):")
print(f"  Count: {len(singleton_customers)}")
if len(singleton_customers) > 0:
    print(f"\n  Vulnerability: If this product is deprecated, customer has no alternative.")
    print(f"  Top single-product customers by value:")
    for _, row in singleton_customers.nlargest(5, 'account_value').iterrows():
        print(f"    {row['company_name']:<35} ${row['account_value']:>12,.0f}")

## 9. What-If Scenarios

In [ ]:
# Scenario 1: Product deprecation
print("Scenario 1: Deprecate a Popular Product")
print("="*60)

popular_products = purchases_df.groupby('product_id').size().reset_index(name='customer_count').nlargest(1, 'customer_count')

if len(popular_products) > 0:
    prod_to_deprecate = popular_products.iloc[0]['product_id']
    affected_custs = purchases_df[purchases_df['product_id'] == prod_to_deprecate]['customer_id'].unique()
    
    affected_revenue = customers_df[customers_df['customer_id'].isin(affected_custs)]['account_value'].sum()
    
    prod_name = products_df[products_df['product_id'] == prod_to_deprecate]['product_name'].values[0]
    
    print(f"\nDeprecating: {prod_name} ({prod_to_deprecate})")
    print(f"  Directly affects: {len(affected_custs)} customers")
    print(f"  Revenue at risk: ${affected_revenue:,.0f}")
    print(f"  Pct of total: {100*affected_revenue/customers_df['account_value'].sum():.1f}%")
    
    # How many have alternatives
    with_alternatives = 0
    for cust in affected_custs:
        products = purchases_df[purchases_df['customer_id'] == cust]['product_id'].tolist()
        if len(products) > 1:
            with_alternatives += 1
    
    print(f"  With alternatives: {with_alternatives}/{len(affected_custs)} ({100*with_alternatives/len(affected_custs):.0f}%)")
    at_risk = len(affected_custs) - with_alternatives
    at_risk_revenue = customers_df[customers_df['customer_id'].isin(
        [c for c in affected_custs if len(purchases_df[purchases_df['customer_id']==c]['product_id'].tolist()) == 1]
    )]['account_value'].sum()
    print(f"  At serious risk (no alternative): {at_risk} customers, ${at_risk_revenue:,.0f}")
print()

In [ ]:
# Scenario 2: Consolidate duplicates
print("Scenario 2: Consolidate Duplicate Accounts")
print("="*60)

if len(dup_df) > 0 and len(dup_df[dup_df['combined_score'] > 0.7]) > 0:
    top_dup = dup_df[dup_df['combined_score'] > 0.7].iloc[0]
    
    combined_value = top_dup['value_1'] + top_dup['value_2']
    combined_products = pd.concat([
        purchases_df[purchases_df['customer_id'] == top_dup['customer_1']]['product_id'],
        purchases_df[purchases_df['customer_id'] == top_dup['customer_2']]['product_id']
    ]).unique()
    
    print(f"\nConsolidating:")
    print(f"  {top_dup['company_1']} ({top_dup['customer_1']}) + {top_dup['company_2']} ({top_dup['customer_2']})")
    print(f"\nImpact:")
    print(f"  Combined revenue: ${combined_value:,.0f}")
    print(f"  Combined products: {len(combined_products)}")
    print(f"  Data quality improvement: Remove 1 duplicate, consolidate {int(top_dup['jaccard']*100)}% overlap")
else:
    print("No high-confidence duplicate pairs to consolidate.")

In [ ]:
# Scenario 3: Cross-sell opportunity
print("\nScenario 3: Cross-Sell Recommendations")
print("="*60)

# For each customer, find products similar customers buy
cross_sell_opps = []

for cust_id in customers_df['customer_id'][:20]:  # Sample first 20
    current_products = set(purchases_df[purchases_df['customer_id'] == cust_id]['product_id'])
    
    # Find similar customers
    similar = similarity_df[
        ((similarity_df['customer_1'] == cust_id) | (similarity_df['customer_2'] == cust_id)) &
        (similarity_df['jaccard'] > 0.3)
    ]
    
    if len(similar) > 0:
        # Get other customer IDs
        other_custs = []
        for _, row in similar.iterrows():
            other = row['customer_2'] if row['customer_1'] == cust_id else row['customer_1']
            other_custs.append(other)
        
        # What do they buy that this customer doesn't?
        other_products = set()
        for other_cust in other_custs:
            other_products.update(purchases_df[purchases_df['customer_id'] == other_cust]['product_id'])
        
        recommended = other_products - current_products
        
        if len(recommended) > 0:
            cust_row = customers_df[customers_df['customer_id'] == cust_id].iloc[0]
            cross_sell_opps.append({
                'customer_id': cust_id,
                'company': cust_row['company_name'],
                'value': cust_row['account_value'],
                'recommended_count': len(recommended)
            })

if cross_sell_opps:
    print(f"\nCross-sell opportunities found: {len(cross_sell_opps)}")
    for opp in sorted(cross_sell_opps, key=lambda x: x['value'], reverse=True)[:5]:
        print(f"  {opp['company']:<30} ${opp['value']:>12,.0f}  ({opp['recommended_count']} products to offer)")
    print(f"\n💡 Potential revenue uplift from cross-sell to top customers")

## 10. Exercises

### Exercise 1: Compute Customer Similarity

In [ ]:
# SOLUTION

def compute_customer_similarity(purchases_df, metric='jaccard'):
    """
    Compute pairwise similarity matrix for all customers.
    
    Args:
        purchases_df: DataFrame with customer_id, product_id columns
        metric: 'jaccard', 'cosine', 'adamic_adar'
    
    Returns:
        DataFrame with columns: customer_1, customer_2, similarity
    """
    customer_ids = purchases_df['customer_id'].unique()
    
    # Create customer→product sets
    cust_products = {}
    for cust_id in customer_ids:
        cust_products[cust_id] = set(purchases_df[purchases_df['customer_id'] == cust_id]['product_id'].tolist())
    
    # Product degrees (for Adamic-Adar)
    product_degrees = purchases_df['product_id'].value_counts().to_dict()
    
    similarities = []
    
    for cust1, cust2 in combinations(customer_ids, 2):
        products1 = cust_products[cust1]
        products2 = cust_products[cust2]
        
        if len(products1) == 0 or len(products2) == 0:
            continue
        
        intersection = len(products1 & products2)
        union = len(products1 | products2)
        
        if metric == 'jaccard':
            score = intersection / union if union > 0 else 0
        elif metric == 'cosine':
            score = intersection / np.sqrt(len(products1) * len(products2))
        elif metric == 'adamic_adar':
            score = 0
            if intersection > 0:
                for product in (products1 & products2):
                    degree = product_degrees.get(product, 1)
                    score += 1 / np.log(degree + 1)
        else:
            score = intersection
        
        similarities.append({
            'customer_1': cust1,
            'customer_2': cust2,
            'similarity': score
        })
    
    return pd.DataFrame(similarities)


print("\nExercise 1: Compute Customer Similarity")
print("="*60)
print(f"\nTest all three metrics:")

for metric in ['jaccard', 'cosine', 'adamic_adar']:
    sim_df = compute_customer_similarity(purchases_df, metric=metric)
    print(f"\n  {metric.upper()}:")
    print(f"    Pairs computed: {len(sim_df)}")
    print(f"    Mean similarity: {sim_df['similarity'].mean():.4f}")
    print(f"    Max similarity: {sim_df['similarity'].max():.4f}")
    print(f"    Pairs with similarity > 0.5: {len(sim_df[sim_df['similarity'] > 0.5])}")

### Exercise 2: Bipartite Projection and Analysis

In [ ]:
# SOLUTION

def project_and_analyze(G, node_type, filter_by_weight=None):
    """
    Project bipartite graph to single mode.
    
    Args:
        G: Bipartite graph
        node_type: 'customer' or 'product' (nodes to project onto)
        filter_by_weight: If specified, only include edges with weight >= this
    
    Returns:
        Projected graph with edge weights
    """
    # Extract nodes of specified type
    nodes = set(n for n in G.nodes() if G.nodes[n].get('node_type') == node_type)
    other_type = 'product' if node_type == 'customer' else 'customer'
    other_nodes = set(n for n in G.nodes() if G.nodes[n].get('node_type') == other_type)
    
    # Project
    G_projected = nx.bipartite.projected_graph(G, nodes, multigraph=False)
    
    # Add weights (shared connections)
    for node1, node2 in G_projected.edges():
        neighbors1 = set(G.neighbors(node1)) & other_nodes
        neighbors2 = set(G.neighbors(node2)) & other_nodes
        weight = len(neighbors1 & neighbors2)
        G_projected[node1][node2]['weight'] = weight
    
    # Filter by weight if specified
    if filter_by_weight:
        edges_to_remove = [(u, v) for u, v, w in G_projected.edges(data='weight') if w < filter_by_weight]
        G_projected.remove_edges_from(edges_to_remove)
    
    return G_projected


print("\nExercise 2: Bipartite Projections")
print("="*60)

# Project to customer graph
G_cust = project_and_analyze(G, 'customer')
print(f"\nCustomer Projection (all edges):")
print(f"  Nodes: {G_cust.number_of_nodes()}")
print(f"  Edges: {G_cust.number_of_edges()}")
print(f"  Density: {nx.density(G_cust):.4f}")

# Project to product graph
G_prod = project_and_analyze(G, 'product')
print(f"\nProduct Projection (all edges):")
print(f"  Nodes: {G_prod.number_of_nodes()}")
print(f"  Edges: {G_prod.number_of_edges()}")
print(f"  Density: {nx.density(G_prod):.4f}")

# Filtered projections
G_cust_strong = project_and_analyze(G, 'customer', filter_by_weight=3)
print(f"\nCustomer Projection (3+ shared products):")
print(f"  Nodes: {G_cust_strong.number_of_nodes()}")
print(f"  Edges: {G_cust_strong.number_of_edges()}")
print(f"  Density: {nx.density(G_cust_strong):.4f}")
print(f"\n✓ Edge filtering reduces noise and reveals strong patterns")

### Exercise 3: Detect Duplicate Candidates

In [ ]:
# SOLUTION

def detect_duplicate_candidates(customers_df, purchases_df, 
                               similarity_threshold=0.75,
                               name_edit_threshold=3,
                               confidence_threshold=0.7):
    """
    Detect likely duplicate customer records using multi-signal approach.
    
    Args:
        customers_df: Customer DataFrame
        purchases_df: Purchase DataFrame
        similarity_threshold: Minimum Jaccard similarity to consider
        name_edit_threshold: Maximum edit distance for name matching
        confidence_threshold: Minimum combined score to report
    
    Returns:
        DataFrame of duplicate candidates ranked by confidence
    """
    # Compute similarities
    sim_df = compute_customer_similarity(purchases_df, metric='jaccard')
    high_sim = sim_df[sim_df['similarity'] > similarity_threshold]
    
    duplicates = []
    
    for _, row in high_sim.iterrows():
        cust1_id = row['customer_1']
        cust2_id = row['customer_2']
        
        cust1 = customers_df[customers_df['customer_id'] == cust1_id].iloc[0]
        cust2 = customers_df[customers_df['customer_id'] == cust2_id].iloc[0]
        
        # Name matching
        edit_dist = fuzzy_string_match(cust1['company_name'], cust2['company_name'])
        name_score = 1 / (1 + edit_dist / name_edit_threshold)
        
        # Industry match
        industry_match = 1 if cust1['industry'] == cust2['industry'] else 0
        
        # Combined score
        confidence = (
            row['similarity'] * 0.5 +
            name_score * 0.3 +
            industry_match * 0.2
        )
        
        if confidence >= confidence_threshold:
            duplicates.append({
                'customer_1': cust1_id,
                'customer_2': cust2_id,
                'company_1': cust1['company_name'],
                'company_2': cust2['company_name'],
                'similarity': row['similarity'],
                'name_distance': edit_dist,
                'industry_match': bool(industry_match),
                'confidence': confidence
            })
    
    return pd.DataFrame(duplicates).sort_values('confidence', ascending=False)


print("\nExercise 3: Detect Duplicate Candidates")
print("="*60)

dup_results = detect_duplicate_candidates(customers_df, purchases_df, 
                                         similarity_threshold=0.70,
                                         confidence_threshold=0.65)

print(f"\nDuplicate candidates detected: {len(dup_results)}")
print(f"\nTop candidates (sorted by confidence):")
print(f"{'Cust 1':<12} {'Cust 2':<12} {'Sim':<8} {'Name Dist':<12} {'Confidence':<12}")
print("-" * 70)

for _, row in dup_results.head(10).iterrows():
    print(f"{row['customer_1']:<12} {row['customer_2']:<12} {row['similarity']:<8.2f} {row['name_distance']:<12} {row['confidence']:<12.3f}")

print(f"\n✓ Exercise complete: Found {len(dup_results)} probable duplicates for review")

## 11. Visualizations

In [ ]:
# Visualization 1: Customer degree distribution (products per customer)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Degree distribution
customer_degrees = metrics_df['products_count'].value_counts().sort_index()
axes[0].bar(customer_degrees.index[:50], customer_degrees.values[:50], 
           color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Products per Customer', fontsize=12)
axes[0].set_ylabel('Number of Customers', fontsize=12)
axes[0].set_title('Customer Product Adoption Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_yscale('log')

# Right: Account value vs products
scatter = axes[1].scatter(metrics_df['products_count'], metrics_df['account_value'], 
                         c=metrics_df['products_count'], cmap='RdYlGn', 
                         s=100, alpha=0.6, edgecolor='black')
axes[1].set_xlabel('Products per Customer', fontsize=12)
axes[1].set_ylabel('Account Value ($)', fontsize=12)
axes[1].set_title('Account Value vs Product Adoption', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization 1: Customer adoption patterns")

In [ ]:
# Visualization 2: Projection comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bipartite degree distribution
bipartite_customer_degrees = [G.degree(c) for c in customer_nodes]
axes[0].hist(bipartite_customer_degrees, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Products per Customer (Bipartite)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Bipartite Degree Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Projection degree distribution
projected_degrees = [G_customer.degree(c) for c in G_customer.nodes()]
axes[1].hist(projected_degrees, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Similar Customers (Projection)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Projected Customer Graph Degree Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Visualization 2: Bipartite vs projected degree distributions")

## Key Takeaways

### Why Bipartite Analysis Matters

Bipartite networks model two-type relationships. Customers aren't intrinsically related—they're connected through products they buy. This "mediated" relationship reveals:

1. **Similarity through shared neighborhoods** (Jaccard, cosine, Adamic-Adar)
2. **Hidden consolidation opportunities** (duplicate detection)
3. **Cross-sell potential** (product co-purchase patterns)
4. **Single points of failure** (niche customers, niche products)
5. **Business value archetypes** (power users, specialists, at-risk)

### When to Use Bipartite Analysis

| Use Case | Method | Output |
|---|---|---|
| Find similar customers | Jaccard/cosine on product sets | Similarity pairs |
| Detect duplicates | Similarity + name + attributes | Consolidation candidates |
| Recommend products | Product co-purchase (projection) | Cross-sell targets |
| Identify vulnerabilities | Degree analysis | Single-expert products |
| Segment customers | Archetype classification | Customer tiers |

### Common Pitfalls

1. **Projection without weights** — Dense projections hide signal; filter by edge weight
2. **Similarity ≠ Duplicate** — High similarity doesn't always mean duplicate; need multi-signal confirmation
3. **Ignoring node types** — Treating bipartite as homogeneous loses structure
4. **Over-relying on one metric** — Combine similarity + attributes + domain knowledge

### Next Lesson

**Lesson 08: Temporal and Process Graphs** — How networks evolve over time. Add timestamps to edges and discover bottlenecks in workflows.